In [1]:
import os
import warnings
from scipy import stats
import numpy as np
import pandas as pd


os.chdir('/Data/EEG-Visual-Experiment') 
warnings.filterwarnings('ignore')

PATTERN_PATH = "./Generated/Spectrums/pattern_morlets.npz"
EXECUTION_PATH = "./Generated/Spectrums/exec_morlets.npz"

from Scripts.Data_Loader import EIRDataset

EIR_P = EIRDataset("./Generated/Data_Pattern/", task_type="geometric", n_jobs=72)
EIR_E = EIRDataset("./Generated/Data_Train/", task_type="geometric", n_jobs=72)

Loading .fif files: 100%|██████████| 840/840 [00:43<00:00, 19.37it/s]


In [8]:
LEFT_CH  = ["P3", "P7", "O1"]
RIGHT_CH = ["P4", "P8", "O2"]
BANDS = {
    "theta": (4, 7),
    "alpha": (8, 12),
    "beta":  (13, 30),
    "gamma": (30, 40)
}

CHANNEL_PAIRS = [
    ("Fp1", "Fp2"),
    ("AF7", "AF8"),
    ("AF3", "AF4"),
    ("F7", "F8"),
    ("F5", "F6"),
    ("F3", "F4"),
    ("F1", "F2"),
    ("FT7", "FT8"),
    ("FC5", "FC6"),
    ("FC3", "FC4"),
    ("FC1", "FC2"),
    ("T7", "T8"),
    ("C5", "C6"),
    ("C3", "C4"),
    ("C1", "C2"),
    ("TP7", "TP8"),
    ("CP5", "CP6"),
    ("CP3", "CP4"),
    ("CP1", "CP2"),
    ("P7", "P8"),
    ("P5", "P6"),
    ("P3", "P4"),
    ("P1", "P2"),
    ("PO7", "PO8"),
    ("PO3", "PO4"),
    ("O1", "O2"),
]
freqs = np.linspace(2, 40, 195)

In [9]:
def count_trials(npz):
    return sum(k.startswith("power_") for k in npz.files)

In [10]:
# def trial_ai(power_chft, freqs, idx_left, idx_right, fmin, fmax):
#     """
#     power_chft: (ch, f, t)
#     1) mean over time
#     2) integrate over freqs in band
#     3) average channels in ROI
#     4) AI = (R-L)/(R+L)
#     """
#     band = (freqs >= fmin) & (freqs <= fmax)
#     p = power_chft[:, band, :].mean(axis=2)              
#     bp = np.trapz(p, freqs[band], axis=1)                 

#     L = float(bp[idx_left].mean())
#     R = float(bp[idx_right].mean())
#     ai = (R - L) / (R + L + 1e-12)
#     return L, R, ai

# For all electrodes:

In [11]:
def trial_ai_pair(power_chft, freqs, idx_left, idx_right, fmin, fmax):
    """
    power_chft: (ch, f, t)

    1) mean over time
    2) integrate over freqs in band
    3) take power in one left-right pair
    4) AI = (R-L)/(R+L)
    """
    band = (freqs >= fmin) & (freqs <= fmax)
    p = power_chft[:, band, :].mean(axis=2)          # (ch, f_band)
    bp = np.trapz(p, freqs[band], axis=1)            # (ch,)

    L = float(bp[idx_left])
    R = float(bp[idx_right])
    ai = (R - L) / (R + L + 1e-12)
    return L, R, ai

In [12]:
# def build_df(npz_path, condition, ch_names, freqs):
#     npz = np.load(npz_path, allow_pickle=True, mmap_mode="r")
#     n = count_trials(npz)

#     freqs = np.asarray(freqs, dtype=float)
#     if len(freqs) != npz["power_0"].shape[1]:
#         raise ValueError(f"freqs length={len(freqs)} but power has n_freq={npz['power_0'].shape[1]}")

#     idx_left  = [ch_names.index(ch) for ch in LEFT_CH  if ch in ch_names]
#     idx_right = [ch_names.index(ch) for ch in RIGHT_CH if ch in ch_names]
#     if not idx_left or not idx_right:
#         raise ValueError(f"ROI channels not found. idx_left={idx_left}, idx_right={idx_right}. Check ch_names.")

#     rows = []
#     for i in range(n):
#         power = npz[f"power_{i}"]                           # (63,195,521)
#         subj  = int(npz[f"subject_id_{i}"].item())
#         trial = int(npz[f"trial_id_{i}"].item())

#         row = {
#             "subject_id": subj,
#             "trial_id": trial,
#             "condition": condition
#         }

#         for band_name, (fmin, fmax) in BANDS.items():
#             L, R, ai = trial_ai(power, freqs, idx_left, idx_right, fmin, fmax)
#             row[f"{band_name}_left"]  = L
#             row[f"{band_name}_right"] = R
#             row[f"{band_name}_ai"]    = ai

#         rows.append(row)

#     return pd.DataFrame(rows)

# For all electrodes:

In [13]:
def build_df_all_pairs(npz_path, condition, ch_names, freqs, channel_pairs):
    npz = np.load(npz_path, allow_pickle=True, mmap_mode="r")
    n = count_trials(npz)

    freqs = np.asarray(freqs, dtype=float)
    if len(freqs) != npz["power_0"].shape[1]:
        raise ValueError(
            f"freqs length={len(freqs)} but power has n_freq={npz['power_0'].shape[1]}"
        )

    available_pairs = []
    missing_pairs = []

    for ch_l, ch_r in channel_pairs:
        if ch_l in ch_names and ch_r in ch_names:
            available_pairs.append((ch_l, ch_r, ch_names.index(ch_l), ch_names.index(ch_r)))
        else:
            missing_pairs.append((ch_l, ch_r))

    print(f"\n[{condition}] Available pairs: {len(available_pairs)}")
    if missing_pairs:
        print(f"[{condition}] Missing pairs: {missing_pairs}")

    rows = []
    for i in range(n):
        power = npz[f"power_{i}"]   # (ch, f, t)
        subj  = int(npz[f"subject_id_{i}"].item())
        trial = int(npz[f"trial_id_{i}"].item())

        base = {
            "subject_id": subj,
            "trial_id": trial,
            "condition": condition
        }

        for ch_l, ch_r, idx_l, idx_r in available_pairs:
            pair_name = f"{ch_l}_{ch_r}"

            row = base.copy()
            row["pair"] = pair_name
            row["left_ch"] = ch_l
            row["right_ch"] = ch_r

            for band_name, (fmin, fmax) in BANDS.items():
                L, R, ai = trial_ai_pair(power, freqs, idx_l, idx_r, fmin, fmax)
                row[f"{band_name}_left"]  = L
                row[f"{band_name}_right"] = R
                row[f"{band_name}_ai"]    = ai

            rows.append(row)

    return pd.DataFrame(rows)

In [14]:
# def paired_test(df_all, ai_col="beta_ai"):
#     df_subj = (df_all.groupby(["subject_id", "condition"], as_index=False)
#                     .agg(ai=(ai_col, "mean"),
#                          n_trials=("trial_id", "count")))

#     wide = df_subj.pivot(index="subject_id", columns="condition", values="ai")
#     common = wide.dropna().index

#     ai_pattern = wide.loc[common, "pattern"]
#     ai_exec    = wide.loc[common, "exec"]

#     t_stat, p_val = stats.ttest_rel(ai_pattern, ai_exec)
#     diff = ai_pattern - ai_exec
#     dz = diff.mean() / diff.std(ddof=1)

#     print(f"\n--- EEG AI ({ai_col}): pattern vs exec (paired by subject) ---")
#     print(f"n_subjects: {len(common)}")
#     print(f"t: {t_stat:.4f}")
#     print(f"p: {p_val:.10f}")
#     print(f"Cohen's dz: {dz:.4f}")

#     if p_val < 0.05:
#         print("Conclusion: H0 rejected (differs between pattern and exec).")
#     else:
#         print("Conclusion: Fail to reject H0 (no significant difference).")

#     return df_subj, wide


In [15]:
# def subj_lr_table(df_all, band_name):
#     Lcol = f"{band_name}_left"
#     Rcol = f"{band_name}_right"

#     df_subj = (df_all.groupby(["subject_id", "condition"], as_index=False)
#                     .agg(L=(Lcol, "mean"),
#                          R=(Rcol, "mean")))
#     return df_subj


# def test_hemisphere_and_phase(df_all, band_name):
#     df_subj = subj_lr_table(df_all, band_name)

#     wideL = df_subj.pivot(index="subject_id", columns="condition", values="L")
#     wideR = df_subj.pivot(index="subject_id", columns="condition", values="R")

#     common = wideL.dropna().index.intersection(wideR.dropna().index)

#     print(f"\n=== {band_name.upper()} ===")
#     print(f"n_subjects (complete): {len(common)}")

#     for cond in ["pattern", "exec"]:
#         R = wideR.loc[common, cond]
#         L = wideL.loc[common, cond]

#         t_stat, p_val = stats.ttest_rel(R, L)
#         diff = (R - L)
#         dz = diff.mean() / (diff.std(ddof=1) + 1e-12)

#         print(f"[Within {cond}] R vs L: t={t_stat:.4f}  p={p_val:.10f}  dz={dz:.4f}")

#     d_pat = (wideR.loc[common, "pattern"] - wideL.loc[common, "pattern"])
#     d_exe = (wideR.loc[common, "exec"]    - wideL.loc[common, "exec"])

#     t_stat, p_val = stats.ttest_rel(d_pat, d_exe)
#     diff = (d_pat - d_exe)
#     dz = diff.mean() / (diff.std(ddof=1) + 1e-12)

#     print(f"[Phase effect on (R-L)] pattern vs exec: t={t_stat:.4f}  p={p_val:.10f}  dz={dz:.4f}")

#     return df_subj


In [16]:
def center_signal(x, method="median"):
    x = np.asarray(x, float)
    c = np.nanmedian(x) if method == "median" else np.nanmean(x)
    return x - c

def li_from_centered_signal(xc, deadzone=None):
    xc = np.asarray(xc, float)
    xc = xc[np.isfinite(xc)]
    if xc.size < 200:
        return np.nan

    if deadzone is None:
        deadzone = 0.1 * np.std(xc) + 1e-12

    right = np.mean(xc >  deadzone)
    left  = np.mean(xc < -deadzone)
    denom = right + left
    return np.nan if denom == 0 else float((right - left) / denom)

def saccade_direction_li_from_eog(xc, sacc, sfreq=1000.0, pre_ms=10, post_ms=20, deadzone=None):
    xc = np.asarray(xc, float)
    sacc = np.asarray(sacc, float)

    n = min(xc.size, sacc.size)
    if n < 500:
        return np.nan
    xc = xc[:n]
    sacc = sacc[:n]

    good = np.isfinite(xc) & np.isfinite(sacc)
    xc = xc[good]
    sacc = sacc[good]
    if xc.size < 500:
        return np.nan

    pre = int(round(pre_ms * sfreq / 1000))
    post = int(round(post_ms * sfreq / 1000))

    edges = np.where((sacc[1:] > 0.5) & (sacc[:-1] <= 0.5))[0] + 1
    if edges.size < 5:
        return np.nan

    if deadzone is None:
        deadzone = 0.05 * np.std(xc) + 1e-12

    dirs = []
    for idx in edges:
        a = max(0, idx - pre)
        b = min(xc.size - 1, idx + post)
        if b <= a + 2:
            continue
        dx = xc[b] - xc[a]
        if abs(dx) <= deadzone:
            continue
        dirs.append(np.sign(dx))

    if len(dirs) < 5:
        return np.nan

    dirs = np.asarray(dirs)
    right = np.mean(dirs > 0)
    left  = np.mean(dirs < 0)
    denom = right + left
    return np.nan if denom == 0 else float((right - left) / denom)

def event_rate(binary, sfreq):
    binary = np.asarray(binary, float)
    binary = binary[np.isfinite(binary)]
    if binary.size < 2:
        return np.nan
    edges = np.where((binary[1:] > 0.5) & (binary[:-1] <= 0.5))[0]
    dur_s = binary.size / sfreq
    return float(edges.size / dur_s) if dur_s > 0 else np.nan

def build_eye_df_from_eirdataset(EIR_Dataset, condition_map=None):
    rows = []
    for i in range(len(EIR_Dataset)):
        eeg_s, eye_s, meta, label, img = EIR_Dataset[i]

        subj = meta.get("subject_id", meta.get("subj_id", meta.get("subject", None)))
        trial = meta.get("trial_id", meta.get("trial", meta.get("epoch_id", i)))

        if subj is None:
            raise KeyError(f"Не нашёл subject_id в metadata. Пример meta: {meta}")

        cond = label
        if isinstance(cond, (np.ndarray, list)):
            cond = cond[0]

        if condition_map is not None:
            cond = condition_map.get(cond, cond)

        cond = str(cond)

        sfreq = eye_s.info["sfreq"]
        x = eye_s.get_data(picks=["EOG_x"]).ravel()
        sacc = eye_s.get_data(picks=["EOG_saccade"]).ravel()
        blink = eye_s.get_data(picks=["EOG_blink"]).ravel()

        xc = center_signal(x, method="median")

        li_eogx = li_from_centered_signal(xc)
        li_sacc = saccade_direction_li_from_eog(xc, sacc, sfreq=sfreq)

        rows.append({
            "subject_id": int(subj),
            "trial_id": int(trial),
            "condition": cond,
            "eogx_li": li_eogx,
            "sacc_li": li_sacc,
            "sacc_rate": event_rate(sacc, sfreq),
            "blink_rate": event_rate(blink, sfreq),
        })

    df_eye = pd.DataFrame(rows)
    print(f"\n[Eye] segments in EIRDataset: {len(df_eye)}")
    return df_eye



def paired_test_generic(df_all, value_col, label):
    df_subj = (df_all.groupby(["subject_id", "condition"], as_index=False)
                    .agg(val=(value_col, "mean"),
                         n_trials=("trial_id", "count")))

    wide = df_subj.pivot(index="subject_id", columns="condition", values="val")
    common = wide.dropna().index

    v_pat = wide.loc[common, "pattern"]
    v_exe = wide.loc[common, "exec"]

    t_stat, p_val = stats.ttest_rel(v_pat, v_exe)
    diff = v_pat - v_exe
    dz = diff.mean() / (diff.std(ddof=1) + 1e-12)

    print(f"\n--- {label}: pattern vs exec (paired by subject) ---")
    print(f"n_subjects: {len(common)}")
    print(f"t: {t_stat:.4f}")
    print(f"p: {p_val:.10f}")
    print(f"Cohen's dz: {dz:.6f}")

    return df_subj, wide


In [ ]:
EIR_for_ch = EIRDataset('./Generated/Data_Train(Exec_and_Rest)/', task_type='all', n_jobs=1)

eeg_s, _, _, _, _ = EIR_for_ch[0]
ch_names = eeg_s.ch_names

# df_pat = build_df(PATTERN_PATH, "pattern", ch_names, freqs)
# df_exe = build_df(EXECUTION_PATH, "exec", ch_names, freqs)
# df_all = pd.concat([df_pat, df_exe], ignore_index=True)

# For all electrodes/////////////////////////
df_pat = build_df_all_pairs(PATTERN_PATH, "pattern", ch_names, freqs, CHANNEL_PAIRS)
df_exe = build_df_all_pairs(EXECUTION_PATH, "exec", ch_names, freqs, CHANNEL_PAIRS)
df_all = pd.concat([df_pat, df_exe], ignore_index=True)

print(df_all.shape)
display(df_all.head())
# //////////////////////

df_eye_p = build_eye_df_from_eirdataset(EIR_P); df_eye_p["condition"] = "pattern"
df_eye_e = build_eye_df_from_eirdataset(EIR_E); df_eye_e["condition"] = "exec"
df_eye = pd.concat([df_eye_p, df_eye_e], ignore_index=True)

df_eye_trial = (df_eye.groupby(["subject_id","trial_id","condition"], as_index=False)
    .agg(eogx_li=("eogx_li","mean"), sacc_li=("sacc_li","mean"),
         sacc_rate=("sacc_rate","mean"), blink_rate=("blink_rate","mean"),
         n_segments=("eogx_li","size"))
)

df_all = df_all.merge(
    df_eye_trial[["subject_id","trial_id","condition","eogx_li","sacc_li","sacc_rate","blink_rate","n_segments"]],
    on=["subject_id","trial_id","condition"], how="left"
)

paired_test_generic(df_all.dropna(subset=["eogx_li"]), "eogx_li", "EOG_x LI (centered)")
paired_test_generic(df_all.dropna(subset=["sacc_li"]), "sacc_li", "Saccade direction LI")
paired_test_generic(df_all.dropna(subset=["sacc_rate"]), "sacc_rate", "Saccade rate (1/s)")
paired_test_generic(df_all.dropna(subset=["blink_rate"]), "blink_rate", "Blink rate (1/s)")

# for band_name in BANDS:
#     paired_test(df_all, ai_col=f"{band_name}_ai")
#     test_hemisphere_and_phase(df_all, band_name)

# For all electrodes:
all_pair_results = []
for band_name in BANDS:
    df_band = paired_stats_by_pair(df_all, band_name)
    all_pair_results.append(df_band)

df_pair_results = pd.concat(all_pair_results, ignore_index=True)
df_pair_results = add_fdr_per_band(df_pair_results)

df_pair_results = df_pair_results.sort_values(["band", "p_fdr", "p"])
display(df_pair_results.head(50))

Loading .fif files: 100%|██████████| 1260/1260 [04:53<00:00,  4.29it/s]



[pattern] Available pairs: 26


# For all electrodes:

In [ ]:
from scipy import stats

def paired_stats_by_pair(df_all, band_name, cond_a="pattern", cond_b="exec"):
    ai_col = f"{band_name}_ai"
    rows = []

    for pair_name, df_pair in df_all.groupby("pair"):
        df_subj = (df_pair.groupby(["subject_id", "condition"], as_index=False)
                          .agg(val=(ai_col, "mean"),
                               n_trials=("trial_id", "count")))

        wide = df_subj.pivot(index="subject_id", columns="condition", values="val")

        if cond_a not in wide.columns or cond_b not in wide.columns:
            continue

        wide = wide[[cond_a, cond_b]].dropna()
        if len(wide) < 3:
            continue

        a = wide[cond_a].astype(float)
        b = wide[cond_b].astype(float)
        diff = a - b

        t_stat, p_val = stats.ttest_rel(a, b, nan_policy="omit")
        dz = float(diff.mean() / (diff.std(ddof=1) + 1e-12))

        rows.append({
            "pair": pair_name,
            "band": band_name,
            "n": int(len(wide)),
            "mean_pattern": float(a.mean()),
            "mean_exec": float(b.mean()),
            "mean_diff": float(diff.mean()),
            "t": float(t_stat),
            "p": float(p_val),
            "dz": float(dz),
        })

    return pd.DataFrame(rows)

In [ ]:
from statsmodels.stats.multitest import fdrcorrection

def add_fdr_per_band(df_res):
    out = []
    for band_name, d in df_res.groupby("band", group_keys=False):
        d = d.copy()
        mask = np.isfinite(d["p"].values)
        pvals = d.loc[mask, "p"].values

        if len(pvals) > 0:
            reject, p_fdr = fdrcorrection(pvals, alpha=0.05, method="indep")
            d.loc[mask, "p_fdr"] = p_fdr
            d.loc[mask, "significant_fdr"] = reject
        else:
            d["p_fdr"] = np.nan
            d["significant_fdr"] = False

        out.append(d)

    return pd.concat(out, ignore_index=True)

In [ ]:
def format_pair_results_table(df):
    df = df.copy()
    for c in ["mean_pattern", "mean_exec", "mean_diff", "t", "dz"]:
        df[c] = df[c].map(lambda x: f"{x:.4f}" if np.isfinite(x) else "nan")
    for c in ["p", "p_fdr"]:
        df[c] = df[c].map(lambda x: f"{x:.4g}" if np.isfinite(x) else "nan")
    df["n"] = df["n"].astype(int).astype(str)
    df["significant_fdr"] = df["significant_fdr"].fillna(False)
    return df

In [ ]:
print("\n=== All electrode pairs: pattern vs exec for AI ===")
display(format_pair_results_table(df_pair_results))

In [ ]:
# from scipy import stats
# import matplotlib.pyplot as plt

# def paired_stats_from_subject_means(df, value_col, cond_a="pattern", cond_b="exec"):
#     df_subj = (df.groupby(["subject_id", "condition"], as_index=False)
#                  .agg(val=(value_col, "mean"),
#                       n_trials=("trial_id", "count")))

#     wide = df_subj.pivot(index="subject_id", columns="condition", values="val")
#     wide = wide[[cond_a, cond_b]].dropna()

#     a = wide[cond_a].astype(float)
#     b = wide[cond_b].astype(float)

#     diff = a - b
#     n = diff.shape[0]

#     t_stat, p_val = stats.ttest_rel(a, b, nan_policy="omit")
#     dz = float(diff.mean() / (diff.std(ddof=1) + 1e-12))

#     se = diff.std(ddof=1) / np.sqrt(n)
#     tcrit = stats.t.ppf(0.975, df=n-1)
#     ci_lo = float(diff.mean() - tcrit * se)
#     ci_hi = float(diff.mean() + tcrit * se)

#     return {
#         "n": int(n),
#         f"mean_{cond_a}": float(a.mean()),
#         f"mean_{cond_b}": float(b.mean()),
#         "mean_diff": float(diff.mean()),
#         "ci95_diff": f"[{ci_lo:.4f}; {ci_hi:.4f}]",
#         "t": float(t_stat),
#         "p": float(p_val),
#         "dz": float(dz),
#     }

# def format_results_table(df_res, p_col="p"):
#     df = df_res.copy()
#     df["p"] = df["p"].map(lambda x: f"{x:.4g}" if np.isfinite(x) else "nan")
#     df["t"] = df["t"].map(lambda x: f"{x:.3f}")
#     df["dz"] = df["dz"].map(lambda x: f"{x:.3f}")
#     for c in [c for c in df.columns if c.startswith("mean_") or c == "mean_diff"]:
#         df[c] = df[c].map(lambda x: f"{x:.4f}")
#     df["n"] = df["n"].astype(int).astype(str)
#     return df
# # ---------- 1) EEG таблица по диапазонам ----------

# eeg_rows = []
# for band_name in ["theta", "alpha", "beta", "gamma"]:
#     col = f"{band_name}_ai"
#     st = paired_stats_from_subject_means(df_all, col, cond_a="pattern", cond_b="exec")
#     eeg_rows.append({
#         "EEG band": band_name,
#         "n": st["n"],
#         "mean_pattern": st["mean_pattern"],
#         "mean_exec": st["mean_exec"],
#         "mean_diff (pat-exec)": st["mean_diff"],
#         "95% CI diff": st["ci95_diff"],
#         "t": st["t"],
#         "p": st["p"],
#         "dz": st["dz"],
#     })

# df_eeg_res = pd.DataFrame(eeg_rows)
# df_eeg_show = format_results_table(df_eeg_res)

# print("\n=== EEG: phase effect on AI (pattern vs exec) ===")
# display(df_eeg_show)

# # ---------- 2) Eye tracking таблица ----------

# eye_metrics = [
#     ("EOG_x LI (centered)", "eogx_li"),
#     ("Saccade direction LI", "sacc_li"),
#     ("Saccade rate (1/s)", "sacc_rate"),
#     ("Blink rate (1/s)", "blink_rate"),
# ]

# eye_rows = []
# for label, col in eye_metrics:
#     dft = df_all.dropna(subset=[col]).copy()
#     st = paired_stats_from_subject_means(dft, col, cond_a="pattern", cond_b="exec")
#     eye_rows.append({
#         "Eye metric": label,
#         "n": st["n"],
#         "mean_pattern": st["mean_pattern"],
#         "mean_exec": st["mean_exec"],
#         "mean_diff (pat-exec)": st["mean_diff"],
#         "95% CI diff": st["ci95_diff"],
#         "t": st["t"],
#         "p": st["p"],
#         "dz": st["dz"],
#     })

# df_eye_res = pd.DataFrame(eye_rows)
# df_eye_show = format_results_table(df_eye_res)

# print("\n=== Eye tracking: phase effect (pattern vs exec) ===")
# display(df_eye_show)